# 01 · One spiking neuron

Build and run single spiking neurons with py-spinnaker2: first a neuron with a constant input, then a neuron driven by input spikes through synapses. Everything runs on the Brian2 simulator. Notebook 02 builds a network from these pieces and runs it on the SpiNNaker2 chip.

Work in pairs. **Try** marks exercises for the pair, **Discuss** marks questions we take up together.

This notebook is a short path through the vendor's tutorials, tailored to the workshop. Each section ends with **Go deeper**: where the content comes from and where to read more.

- Vendor tutorials (SpiNNcloud, Apache-2.0): [py-spinnaker2 `tutorials/deep_dive`](https://gitlab.com/spinnaker2/py-spinnaker2/-/tree/v0.8.2/tutorials/deep_dive), referred to as *tutorial 00*, *01*, …
- API documentation: [spinnaker2.gitlab.io/py-spinnaker2](https://spinnaker2.gitlab.io/py-spinnaker2/)
- **Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spinnaker2 import brian2_sim, snn


def backend(hardware=False):
    """A fresh backend for every run: the Brian2 simulator, or one 48-chip SpiNNaker2 board."""
    if hardware:
        from spinnaker2 import hardware as hw

        return hw.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()

**Go deeper:** a network consists of four building blocks: populations, projections, a network and a backend. Brian2 and the chip share the same `run(net, steps)` method, so the code is written once (tutorial 00, sections 3–4).

## 1. One neuron with a constant input

The `lif` neuron of SpiNNaker2 updates its membrane potential once per time step:

$$v[t] = \alpha \, v[t-1] + I_\text{syn}[t] + i_\text{offset}$$

- `alpha_decay` ($\alpha$): how much of the potential is kept per step (0.9 = 10 % leaks away)
- `i_offset`: a constant input added every step
- `threshold`: when $v$ exceeds it, the neuron spikes and $v$ is reset
- `reset`: `"reset_by_subtraction"` subtracts the threshold and keeps the overshoot; `"reset_to_v_reset"` sets $v$ to `v_reset`

The cell below runs one neuron for 50 steps and plots its membrane potential. The recorded potential is the value after a possible reset, so the moment of crossing the threshold shows up as a drop, marked with a red triangle.

<details><summary>Where the equation comes from</summary>

**The membrane as an RC circuit.** The cell membrane separates charge like a capacitor $C$, and ion channels let charge leak through it like a resistor $R$. An input current $I$ splits into a part that charges the capacitor and a part that leaks through the resistor. Writing that down (Kirchhoff's current law) and multiplying by $R$ gives the LIF equation in continuous time (Eq. 5 in the workshop note [Spiking Neurons and Networks](../01_knowledge_base/01_primer_snn_basics.md#24-the-lif-equation-term-by-term)):

$$\tau_m \frac{dV}{dt} = -(V - V_\text{rest}) + R\,I(t), \qquad \tau_m = RC$$

The first term is the leak: it pulls $V$ back to the resting potential $V_\text{rest}$. The second term is the input: it pushes $V$ up. The time constant $\tau_m$ says how fast the neuron forgets.

**From continuous to discrete time.** The chip advances in steps of $\Delta t$ = 1 ms. Measuring $v$ relative to rest ($v = V - V_\text{rest}$) and solving the equation exactly over one step gives (Eq. 9 in the same note, [section 3](../01_knowledge_base/01_primer_snn_basics.md#3-lif-in-discrete-time-as-on-the-chip)):

$$v[t] = \alpha \, v[t-1] + I_\text{syn}[t] + i_\text{offset}, \qquad \alpha = e^{-\Delta t / \tau_m}$$

Without input, $v$ decays exponentially, so over one step it shrinks by the fixed factor $\alpha$. The input of the step is added on top: spikes arriving in the step ($I_\text{syn}$) and a constant input ($i_\text{offset}$). So `alpha_decay = 0.9` corresponds to $\tau_m \approx 9.5$ ms.

Complete derivations in the workshop note *Spiking Neurons and Networks*: [the LIF equation, term by term](../01_knowledge_base/01_primer_snn_basics.md#24-the-lif-equation-term-by-term) (Eq. 2–5) and [LIF in discrete time](../01_knowledge_base/01_primer_snn_basics.md#3-lif-in-discrete-time-as-on-the-chip) (Eq. 9–14). Textbook treatment: [Gerstner et al., *Neuronal Dynamics*, section 1.3](https://neuronaldynamics.epfl.ch/online/Ch1.S3.html).

</details>

In [ ]:
params = {"threshold": 1.0, "alpha_decay": 0.9, "i_offset": 0.15, "reset": "reset_by_subtraction"}

neuron = snn.Population(size=1, neuron_model="lif", params=params, name="neuron", record=["spikes", "v"])
net = snn.Network("one_neuron")
net.add(neuron)
backend().run(net, 50)

v = neuron.get_voltages()[0]
spikes = neuron.get_spikes()[0]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(v, ".-", label="membrane potential v")  # one dot per time step
ax.axhline(params["threshold"], ls="--", c="0.5", label="threshold")
ax.plot(spikes, [params["threshold"]] * len(spikes), "rv", label="spike")
ax.set_xticks(range(51), minor=True)  # one tick per time step: the simulation is discrete
ax.set_xticks(range(0, 51, 5))
ax.set(xlabel="time step (1 step = 1 ms)", ylabel="v (units of threshold)")
ax.legend(loc="lower right")
plt.show()
print(f"{len(spikes)} spikes in 50 steps")

**Try (before scrolling down):** the next cell shows the answers, so please don't peek before you have your own. Work with the cell above.

1. Set `i_offset` to 0.05. Why does the neuron never spike? *Hint:* without spikes, $v$ approaches $i_\text{offset} / (1 - \alpha)$.
2. Work out on paper the smallest `i_offset` that makes the neuron spike with `alpha_decay = 0.9`. Then check it in the cell above.
3. Predict how the number of spikes in 1,000 steps changes as `i_offset` grows from 0 to 0.4. Sketch the curve: where does it start, does it jump, is it straight or bent?

**Spike count vs. input.** The cell below runs 81 neurons at once, each with its own `i_offset` from 0 to 0.4 in steps of 0.005, all other parameters as above, and counts their spikes in 1,000 steps. The red dot is the neuron of the figure above. Compare with your sketch.

In [ ]:
i_offsets = np.round(np.arange(0, 0.41, 0.005), 3)  # one value per neuron
pop = snn.Population(size=len(i_offsets), neuron_model="lif", params={**params, "i_offset": list(i_offsets)}, name="neurons", record=["spikes"])
net = snn.Network("input_output")
net.add(pop)
steps = 1000
backend().run(net, steps)

counts = [len(pop.get_spikes().get(i, [])) for i in range(len(i_offsets))]
critical = (1 - params["alpha_decay"]) * params["threshold"]  # below this input, v never reaches the threshold

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(i_offsets, counts, "o", color="black", ms=4)
nearest = np.argmin(np.abs(i_offsets - params["i_offset"]))  # the neuron closest to the figure above
ax.plot(i_offsets[nearest], counts[nearest], "o", color="red", ms=7)
ax.axvline(critical, ls="--", c="0.5")
ax.text(critical, max(counts), " critical input", va="top", color="0.4")
ax.set(xlabel="i_offset (units of threshold per step)", ylabel=f"spikes in {steps} steps")
plt.show()

<details><summary>Why the curve has small steps</summary>

The neuron can only fire at the tick of a clock, once per time step. So it cannot fire every 3.2 steps; it can only fire in patterns of whole steps, such as "every 3 steps" or "3 times in 10 steps".

Raise the input a little, and the neuron would like to fire a little earlier. But earlier means a whole step earlier, and the small extra input isn't enough for that. So it keeps firing on the same steps as before, and the count stays the same: a flat step in the curve.

The leak makes this stick. A bit of extra charge doesn't pile up, because part of it leaks away every step. Only when the extra input is big enough to reach the threshold a whole step earlier does the neuron switch to a faster pattern, and the count jumps.

Without a leak, the extra charge would pile up until it produced an extra spike, and the count would rise evenly, without steps.

</details>

**Try:** set `alpha_decay` to 0.8 in the first cell of this section and run both cells again. Where does the critical input move, and why?

**Discuss:** the spike count as a function of the input is the neuron's input–output curve. How does it compare with an activation function such as ReLU?

**Go deeper:** tutorial 01, section 3.2 traces the same curve (the f–I curve) by giving every neuron its own `i_offset`; section 3 explains shared, per-neuron and global parameters. All parameters of `lif`: [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html#leaky-integratefire-neuron-lif). Workshop notes: [from Hodgkin-Huxley to LIF](../01_knowledge_base/01_primer_snn_basics.md#2-from-hodgkin-huxley-to-lif) and [LIF in discrete time, as on the chip](../01_knowledge_base/01_primer_snn_basics.md#3-lif-in-discrete-time-as-on-the-chip).

## 2. Input spikes and synapses

Neurons usually get their input as spikes from other neurons. A `spike_list` population sends spikes at given time steps. A `Projection` connects it to the `lif` neuron; each connection is `[pre neuron, post neuron, weight, delay]`.

On the chip, `lif` weights are integers from −15 to 15 and delays from 0 to 7 steps, so the threshold is set to 10 here instead of 1. A spike arrives `delay + 1` steps after it was sent.

<details><summary>Why integers, and why −15 to 15</summary>

Every synapse is stored in the small local memory of a processor core as one packed word with fields for target neuron, delay and weight. For `lif`, the weight field holds a signed integer from −15 to 15 and the delay field 3 bits (0…7). Non-integer weights are rounded toward zero and values outside the range are clipped when the network is mapped, without an error. Variants of the model trade these budgets: `lif_no_delay` drops the delay for 8-bit weights, `lif_32bit` has 20-bit weights. Only the ratio of weight to threshold matters, so a threshold of 10 with weight 5 behaves like a threshold of 1 with weight 0.5. Source: tutorial 02, section 2, and tutorial 01, section 2.

</details>

<details><summary>Why delay + 1</summary>

A spike sent in step $t$ is processed by the receiving core in step $t+1$ at the earliest; `delay` adds steps on top, during which the spike waits in a buffer at the receiving core. With `delay = 0` the post neuron sees the spike one step later. Brian2 reproduces this, so both backends give the same spike times. Source: tutorial 02, section 2 (delay line demo).

</details>

In [ ]:
input_times = {0: [5, 7, 9], 1: [30], 2: [32]}  # input neuron -> spike times

inp = snn.Population(size=3, neuron_model="spike_list", params=input_times, name="input")
post = snn.Population(
    size=1,
    neuron_model="lif",
    params={"threshold": 10.0, "alpha_decay": 0.8, "reset": "reset_by_subtraction"},
    name="post",
    record=["spikes", "v"],
)
weight = 5  # same weight for all three input neurons, half the threshold of 10
conns = [[i, 0, weight, 0] for i in range(3)]  # pre, post, weight, delay
net = snn.Network("synapses")
net.add(inp, post, snn.Projection(pre=inp, post=post, connections=conns))
backend().run(net, 50)

post_spikes = post.get_spikes()[0]
fig, (ax_in, ax_v, ax_out) = plt.subplots(3, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [1, 3, 1]})

# top: input spikes, one row per input neuron
ax_in.eventplot([input_times[i] for i in range(3)], colors="grey", lineoffsets=range(3), linelengths=0.7)
ax_in.set(yticks=range(3), yticklabels=[f"input {i}" for i in range(3)], ylim=(-0.6, 2.6), title="input spikes")

# middle: membrane potential of the post neuron, one dot per time step
ax_v.plot(post.get_voltages()[0], ".-")
ax_v.axhline(10.0, ls="--", c="0.5")
ax_v.text(0, 10.2, "threshold", va="bottom", color="0.4")
ax_v.set(ylabel="v (threshold = 10)", title="membrane potential of the post neuron")

# bottom: output spikes of the post neuron
ax_out.eventplot([post_spikes], colors="black", lineoffsets=0, linelengths=0.7)
ax_out.set(yticks=[0], yticklabels=["post"], ylim=(-0.6, 0.6), title="output spikes")

ax_out.set_xticks(range(51), minor=True)  # one tick per time step
ax_out.set_xticks(range(0, 51, 5))
ax_out.set(xlim=(-0.5, 50.5), xlabel="time step (1 step = 1 ms)")
plt.tight_layout()
plt.show()
print("post neuron spikes at:", post_spikes)

**Try:**

1. One input spike is not enough to reach the threshold. Which input makes the neuron spike: the three spikes of neuron 0, or the two spikes of neurons 1 and 2? Why?
2. Change `alpha_decay` to 0.95 and to 0.5. What happens?
3. Make one connection inhibitory (negative weight). Where does the potential go?

**Discuss:** in a deep network, the order and timing of the inputs don't matter for a neuron's output. Here they do. What does the leak turn the neuron into?

**Go deeper:** tutorial 00, section 5 builds the same kind of network with one excitatory and one inhibitory input and explains both reset rules. Tutorial 02, section 2 shows the `delay + 1` rule with a delay line. Workshop notes: [synapses](../01_knowledge_base/01_primer_snn_basics.md#4-synapses) and [where Brian2 and the chip differ](../01_knowledge_base/03_brian2_fallback.md#3-differences-from-the-chip).

---

**End of notebook 01.** Back to the slides on the SpiNNaker2 hardware; notebook 02 follows after that block.